# Epsilon Fund — XS Momentum (Long-Side Volume Filter, Unfiltered Short + Inverse-Vol Weighting)

Variant of `xs_3_3.ipynb` (long-side volume filter only) with **inverse-
volatility weighting within each leg** added on top.  Cousin of `xs_3_2_i`
that drops the short-side stage-2 refinement entirely.

Layered ingredients:
- Residual-Sharpe composite signal (`xs_3` foundation)
- Long-leg-only volume interaction: high vol_change refinement on the long
  pool; short leg uses plain bottom-by-composite (no volume stage)
- Intra-leg inverse-vol weighting (as in `xs_3_i` / `xs_3_2_i`)

**A/B map across the inverse-vol family:**
- `xs_3_i`     → no volume filter on either leg + inverse-vol
- `xs_3_2_i`   → high vol_change BOTH legs + inverse-vol
- `xs_3_1_i`   → high vol_change long, low vol_change short + inverse-vol
- `xs_3_3_i`   → high vol_change long, **no filter** short + inverse-vol

If the long-only volume filter beats the symmetric one, the right design is
to keep the volume confirmation on accumulation (long) and let composite
ranking do the work on distribution (short), where vol_change is noisier.

Position weighting as in `xs_3_i`: `1/σ_coin` normalised within each leg.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# ── repo root — works on both Mac and Windows ────────────────────────────────
ROOT = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = os.path.join('C:\\', 'Users', 'user', 'Documents', 'Epsilon Fund', 'Epsilon-Quant-Research')
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.insert(0, os.path.join(ROOT, 'infrastructure', 'backtester'))
sys.path.insert(0, os.path.join(ROOT, 'topics', 'momentum', 'xs_momentum'))
sys.path.insert(0, os.path.join(ROOT, 'topics', 'momentum', 'xs_momentum', 'universe'))

from binance_client  import get_binance_client
from wf_engine       import (walk_forward, plateau_analysis, plateau_summary,
                              perturbation_test, cost_stress_test)
from wf_visualizer   import plot_walk_forward_results, plot_plateau_analysis
from ls_diagnostics  import compute_attribution, plot_attribution
from engine          import backtest
from xs_data         import load_xs_data, build_close_panel, DEFAULT_COINS
from universe_filter import load_cache, get_universe, precompute_avg_volume

---
## Data

Two modes, controlled by `USE_DYNAMIC_UNIVERSE` in the next cell:

**Static mode (`False`)** — fast iteration with a hardcoded coin list (`DEFAULT_COINS`).
Pulls daily OHLCV directly from Binance via `load_xs_data()`. Good for quick parameter
sweeps where the universe is fixed.

**Dynamic mode (`True`)** — universe is re-selected at each rebalance from a local
parquet cache built by `universe_cache.py`. The cache holds **USDT perpetual futures
data only** — perp prices feed both the ranking signal and the backtest PnL (matching
what would actually trade on Hyperliquid for the dollar-neutral L/S strategy). At each
rebalance, `get_universe(dates[r-1], ...)` filters by 30-day avg perp volume and listing
age — strictly no lookahead since only data up to the signal formation date is used.

**Cache maintenance:** run `python universe_cache.py` once to build the cache, then
once per day to incrementally append yesterday's candle. Notebook runs make zero
API calls — they read parquet directly.

**`COST`** — per-leg trading cost fraction. The strategy returns gross
`strategy_returns` plus a `turnover` column; the engine applies
`trade_cost = turnover × COST` automatically. Adjust `COST` without re-running
the strategy — `cost_stress_test()` rescales it for free.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODE
# ══════════════════════════════════════════════════════════════════════════════
USE_DYNAMIC_UNIVERSE = True   # True  → load from perp cache (universe_cache.py)
                               # False → fetch API with COINS list below

# ══════════════════════════════════════════════════════════════════════════════
# UNIVERSE FILTER CONFIG  (used when USE_DYNAMIC_UNIVERSE = True)
# ══════════════════════════════════════════════════════════════════════════════
UF_TOP_N          = 30          # max coins in the eligible universe at each rebalance
UF_MIN_VOLUME     = 80_000_000  # min 30-day avg daily USDT perp volume ($)
UF_MIN_AGE_DAYS   = 90         # min days since listing (perp onboard date)
UF_VOLUME_WINDOW  = 30          # rolling window for avg volume computation

# ══════════════════════════════════════════════════════════════════════════════
# STATIC CONFIG  (used when USE_DYNAMIC_UNIVERSE = False)
# ══════════════════════════════════════════════════════════════════════════════
COINS    = DEFAULT_COINS   # ['BTC','ETH','SOL','BNB','XRP','DOGE','ADA','AVAX','LINK','MATIC']
INTERVAL = '1d'
LOOKBACK = 2200            # days; covers ~6 years for most coins

# ══════════════════════════════════════════════════════════════════════════════
# SHARED
# ══════════════════════════════════════════════════════════════════════════════
COST            = 0.001   # per-leg trading cost fraction
POOL_MULTIPLIER = 2.0     # Lee-Swaminathan two-stage selection: candidate pool
                          # size = POOL_MULTIPLIER × final selection size.
                          # 2.0 = top-2N by composite, then volume picks N.
                          # Higher = volume has more say; lower = composite dominates.



# ── Inverse-vol weighting (intra-leg risk parity) ────────────────────────
# Within each long / short basket, weight each coin by 1/σ_coin (normalised
# so leg weights sum to 1).  σ_coin = rolling stdev of daily returns over
# IV_VOL_WINDOW bars.  Reduces single-coin tail risk (memecoin pumps) and
# brings risk contributions toward parity within each leg.  Literature
# (Moreira-Muir 2017, Asness/Frazzini/Pedersen): adds 0.1-0.3 Sharpe and
# cuts max drawdown ~20-40% on momentum baskets vs equal-weight.
IV_VOL_WINDOW = 30   # days; 30 matches the universe-filter volume window

# ── Static data load (API, only when USE_DYNAMIC_UNIVERSE = False) ────────────
if not USE_DYNAMIC_UNIVERSE:
    client = get_binance_client()
    print('Fetching daily OHLCV...')
    data = load_xs_data(coins=COINS, interval=INTERVAL, lookback=LOOKBACK, client=client)

    panel    = build_close_panel(data)
    volume   = None    # signals no dynamic filtering inside the strategy
    meta     = None

    ew_equity = (1 + panel.pct_change().mean(axis=1).fillna(0)).cumprod()
    ew_df     = pd.DataFrame({'Close': ew_equity}, index=panel.index)
    driver_df = data['BTC'][['Close']].copy()

    print(f'\nClose panel: {panel.shape}  ({panel.index[0].date()} → {panel.index[-1].date()})')
    print(f'Driver (BTC): {len(driver_df)} bars')

In [ ]:
if USE_DYNAMIC_UNIVERSE:
    # ── Load from parquet cache (no API calls) ────────────────────────────────
    # Run  python universe_cache.py  first to build / update the cache.
    print('Dynamic universe mode — loading perp cache...')
    close_full, volume, meta = load_cache()

    print(f'  close  : {close_full.shape}')
    print(f'  volume : {volume.shape}')
    print(f'  meta   : {meta.shape}')

    # ── Clip price panel to LOOKBACK days from the most recent cached date ───
    # `volume` and `meta` intentionally KEEP full history so get_universe's
    # rolling 30-day avg has enough bars even when as_of_date sits at the
    # start of the LOOKBACK window.  Slicing by `as_of_date <= dates[r-1]`
    # inside get_universe still prevents lookahead.
    cutoff     = close_full.index[-1] - pd.Timedelta(days=LOOKBACK)
    close_full = close_full.loc[close_full.index >= cutoff]

    # ── Restrict panel to coins with a recorded listing date in meta ─────────
    tradeable = sorted([c for c in close_full.columns
                         if c in meta.index and pd.notna(meta.loc[c, 'listing_date'])])
    panel = close_full[tradeable]

    # ── Equal-weight basket benchmark ────────────────────────────────────────
    ew_equity = (1 + panel.pct_change().mean(axis=1).fillna(0)).cumprod()
    ew_df     = pd.DataFrame({'Close': ew_equity}, index=panel.index)

    # ── Driver (BTC perp — provides the walk-forward date index) ─────────────
    driver_df = panel[['BTC']].rename(columns={'BTC': 'Close'}).copy()

    print(f'\nTradeable universe: {len(tradeable)} coins')
    print(f'Date range: {panel.index[0].date()} → {panel.index[-1].date()}  '
          f'({len(panel)} bars, clipped to LOOKBACK={LOOKBACK})')
    print(f'volume kept full ({len(volume)} bars) for clean rolling-avg lookback')
else:
    print('Static mode — data already loaded in Cell 3.')

---
## ⚠️  WF Engine Interface — Multi-Asset Adapter

### What the engine expects

`walk_forward()` was designed for single-asset strategies:
```
strategy_fn(df_slice, params)  →  (df, indicator_cols)
```
where `df_slice` is **one coin's** OHLCV for the training or test window, and the
returned `df` must carry a `position` column (`1`/`0`/`-1`). The engine slices a single
`df` by bar index to build train/test windows.

### Why XS momentum doesn't fit

1. **Multi-asset data** — `strategy_fn` receives only one coin's slice, but XS
   momentum needs all coins simultaneously for cross-sectional ranking.
2. **Portfolio returns, not a single position** — the output is a weighted long/short
   portfolio return; there is no single `1`/`0`/`-1` column that makes sense.

### The adapter used here (no engine modifications)

**Closure + `strategy_returns` path:**

- `make_xs_strategy(full_panel, cost_per_leg)` captures the full multi-asset close
  panel as a Python closure.
- Inside `strategy_fn`, `df_slice` is ignored **except for its `DatetimeIndex`**, which
  is used to window `full_panel` to the correct train/test dates.
- The strategy computes per-bar portfolio returns (including transaction costs) and
  stores them in a `strategy_returns` column.
- `position = 1` is set everywhere, activating the engine's precomputed-returns path:
  ```python
  # engine.py — precomputed path
  net_returns  = position.shift(1) * strategy_returns   # bar-0 zeroed (see below)
  equity_curve = (1 + net_returns).cumprod()
  ```
- `cost=0` is passed to `walk_forward` because costs are **already baked** into
  `strategy_returns`. Passing any non-zero cost would double-count.

### Three consequences to be aware of

**1. Bar-0 of every slice has zero return.**  
The engine applies `eff_pos = position.shift(1)`, so `eff_pos[0] = 0` regardless of
`position[0] = 1`. This zeros bar-0's return in every train and test window. On a
daily strategy with 700+ bars per fold the effect is negligible (~0.14% of bars).

**2. `num_trades = 0` — custom `reject_fn` is required.**  
The engine detects trades via `position.diff()`. With `position=1` always,
`position.diff() = 0` everywhere, so no trades are logged. The **default** `reject_fn`
has `if num_trades < 7: return True` which would reject every Optuna trial. The
custom `reject_fn` below uses Sharpe and drawdown filters instead.

**3. `cost_stress_test()` is replaced by a manual version.**  
The built-in `cost_stress_test()` re-runs the OOS DataFrame at higher costs by
re-calling `engine.backtest()`, but because `position=1` always the engine sees no
position changes and adds zero cost. The manual stress test re-runs `strategy_fn`
at escalating `cost_per_leg` values on the OOS date window instead.

### What a proper multi-asset engine would need

If the XS strategy family grows, the engine would need these targeted changes:

| Change | File | Where |
|--------|------|-------|
| Accept `data_dict` alongside `df` so folds slice all coins | `wf_engine.py` | `walk_forward()` signature |
| Pass the full per-coin slice dict to `strategy_fn` | `wf_engine.py` | `_make_objective()` and fold loop |
| Allow `strategy_fn` to return a position *matrix* (bars × coins) | `wf_engine.py` | return contract |
| Aggregate portfolio returns from the position matrix before calling `backtest()` | `wf_engine.py` | `_run_backtest()` |

The closure adapter keeps the engine untouched and is fully sufficient for a
single XS strategy. Consider the engine changes only if you need the WF framework
to also optimise per-coin position sizing or dynamic universe selection.

---

---
## Parameter Configuration

Define which parameters to optimise and which to anchor.

**First run:** leave `FIXED_PARAMS` empty — let Optuna search all three dimensions
freely. After the stability table prints, copy parameters with CV < 0.15 into
`FIXED_PARAMS` and re-run.

| Param | Range | Notes |
|---|---|---|
| `J` | 7–30 days | Standard daily momentum formation window (1 week to 1 month) |
| `K` | 7–14 days | Holding period; shorter = more rebalances = more costs |
| `top_n` | 2–4 coins | Integer; 3 means 3 long + 3 short out of 10 coins (30% each leg) |

In [ ]:
# ── search ranges ─────────────────────────────────────────────────────────────
# Format: 'param_name': ('int' | 'float', lo, hi)
#
# pct_long / pct_short = fraction of the eligible universe to put in each leg.
# Inside the strategy: n_long  = max(1, int(pct_long  * universe_size))   (floor)
#                     n_short = max(1, int(pct_short * universe_size))   (floor)
# Decile portfolios (10%) are the academic standard for XS momentum;
# 20% is a common variant. Beyond ~30% the ranking signal degrades fast.
PARAM_DEFS = {
    'J':         ('int',   7,    45),    # formation / lookback window (days)
    'K':         ('int',   7,    21),    # holding period (days)
    'pct_long':  ('float', 0.05, 0.25),  # fraction of universe in long leg
    'pct_short': ('float', 0.05, 0.25),  # fraction of universe in short leg
}

# ── fixed params ──────────────────────────────────────────────────────────────
# Start empty — anchor stable parameters (CV < 0.15) after the first stability run.
FIXED_PARAMS = {}

# Example after a stability run:
# FIXED_PARAMS = {
#     'pct_long':  0.20,
#     'pct_short': 0.20,
# }

---
## Strategy

**Composite signal + symmetric volume filter (unchanged from `xs_3_2`):**
- Composite = rolling Sharpe of BTC-stripped residuals over J.
- Pool = top `pool_multiplier × n_long` and bottom `pool_multiplier × n_short`
  by composite.
- BOTH legs refine by HIGHEST `vol_change` (rising volume confirms move on
  either side — Lee-Swaminathan 2000).

**Position weighting (NEW — inverse-vol):**
- After final selection, each coin in a leg gets weight `(1/σ_coin) / Σ(1/σ_other_in_leg)`.
- σ_coin = rolling 30-day stdev of daily returns at formation date.
- Leg balance still 50/50; only intra-leg allocation changes.

**Edge cases:** same as `xs_3_i` (NaN vol → weight 0; NaN return mid-hold →
basket renormalises pro-rata).


In [ ]:
def make_xs_strategy(panel, volume=None, meta=None,
                     uf_top_n=None, uf_min_volume=None, uf_min_age=None,
                     uf_volume_window=None,
                     pool_multiplier=2.0,
                     iv_vol_window=30):
    """
    Factory that captures the perp data via closure.
    Returns a strategy_fn compatible with wf_engine.walk_forward().

    Single perp panel for both signal and PnL — perp prices feed the ranking
    signal and the backtest execution.  This matches what would actually trade
    on Hyperliquid for the dollar-neutral L/S strategy.

    Performance
    -----------
    Two-level optimisation for fast Optuna runs:
      1. Rolling avg-volume panel pre-computed ONCE here (shared across all trials)
         → O(1) lookup inside get_universe instead of recomputing rolling mean.
      2. Inner bar loop fully vectorised via numpy:
         - perp_rets converted to a numpy matrix once per call
         - Long / short basket returns computed for the whole K-bar holding
           window at once (np.nanmean over coin axis), bulk-assigned to
           pre-allocated numpy arrays.
      Combined effect: ~10–20× speedup vs the naive implementation.

    Output columns:
      strategy_returns, turnover, long_ret, short_ret, universe_size

    Dynamic universe (no lookahead): when volume and meta are provided,
    get_universe(dates[r-1], ...) is called at every rebalance.

    Turnover:
      long_turn  = |new_long  − old_long|  / n_long    ∈ [0, 1]
      short_turn = |new_short − old_short| / n_short   ∈ [0, 1]
      turnover   = long_turn + short_turn               ∈ [0, 2]
    """
    # ── Resolve filter defaults once ──────────────────────────────────────────
    _top_n      = uf_top_n         if uf_top_n         is not None else len(panel.columns)
    _min_vol    = uf_min_volume    if uf_min_volume    is not None else 50_000_000
    _min_age    = uf_min_age       if uf_min_age       is not None else 180
    _vol_window = uf_volume_window if uf_volume_window is not None else 30

    # ── Pre-compute rolling avg volume ONCE (shared across all Optuna trials) ─
    avg_vol_panel = (precompute_avg_volume(volume, volume_window=_vol_window)
                     if volume is not None else None)

    # ── Pre-compute coin → column-index mapping ──────────────────────────────
    coin_to_idx = {c: i for i, c in enumerate(panel.columns)}

    # ── Pre-compute rolling vol panel (intra-leg inverse-vol weighting) ─────
    # Daily-return stdev over iv_vol_window bars.  Computed once on the full
    # panel; reindexed inside strategy_fn per fold.  shift(1) inside the
    # rebalance loop ensures the vol used at bar r is computed from data
    # through bar r-1 only (no lookahead).
    vol_full = panel.pct_change().rolling(iv_vol_window, min_periods=10).std()

    def strategy_fn(df_slice, params):
        J         = int(params['J'])
        K         = int(params['K'])
        pct_long  = float(params['pct_long'])
        pct_short = float(params['pct_short'])

        # ── Window panel to this fold's date range ────────────────────────────
        dates = df_slice.index
        win   = panel.reindex(dates)
        n     = len(dates)

        # ── Composite signal: rolling Sharpe of residual returns (xs_3 base) ──
        daily_coin  = win.pct_change()
        daily_btc   = win['BTC'].pct_change()
        beta        = daily_coin.rolling(J).cov(daily_btc).divide(
            daily_btc.rolling(J).var(), axis=0
        )
        daily_resid = daily_coin.subtract(beta.multiply(daily_btc, axis=0))
        mom_signal  = (
            daily_resid.rolling(J).mean() / daily_resid.rolling(J).std()
        ).shift(1)

        # ── Lee-Swaminathan volume change signal (LONG LEG ONLY) ─────────
        # vol_change = avg_volume(last 7d) / avg_volume(prior 30d), shifted 1 bar.
        # Used only as the long-side stage-2 refinement.  Short leg is unfiltered.
        if volume is not None:
            vol_win    = volume.reindex(dates)
            vol_change = (vol_win.rolling(7).mean() /
                          vol_win.rolling(30).mean()).shift(1)
        else:
            vol_change = None
        rets_arr   = win.pct_change().values            # numpy view for fast slicing

        # Inverse-vol weighting: per-coin vol at each bar (for weight derivation)
        vol_arr    = vol_full.reindex(dates).values     # shape (n, n_coins)

        # Pre-allocated output arrays
        strategy_returns = np.full(n, np.nan)
        long_returns     = np.full(n, np.nan)
        short_returns    = np.full(n, np.nan)
        turnover_arr     = np.zeros(n)
        universe_size    = np.zeros(n)
        long_turn_arr    = np.zeros(n)   # per-rebalance long-leg turnover  ∈ [0, 1]
        short_turn_arr   = np.zeros(n)   # per-rebalance short-leg turnover ∈ [0, 1]
        n_long_arr       = np.zeros(n)   # actual # coins in long leg per rebalance
        n_short_arr      = np.zeros(n)   # actual # coins in short leg per rebalance

        prev_long  = []
        prev_short = []

        # ── Rebalance loop ────────────────────────────────────────────────────
        for r in range(J + 1, n, K):
            signal = mom_signal.iloc[r]

            # Dynamic universe: eligible coins at dates[r-1] (signal formation)
            if volume is not None and meta is not None:
                eligible = get_universe(
                    as_of_date     = dates[r - 1],
                    volume         = volume,
                    meta           = meta,
                    top_n          = _top_n,
                    min_avg_volume = _min_vol,
                    min_age_days   = _min_age,
                    volume_window  = _vol_window,
                    avg_vol_panel  = avg_vol_panel,
                )
                eligible = [c for c in eligible if c in coin_to_idx]
                if eligible:
                    signal = signal[eligible]

            valid = signal.dropna()

            # ── Percentile → absolute counts (floor, min 1) ───────────────
            n_long  = max(1, int(pct_long  * len(valid)))
            n_short = max(1, int(pct_short * len(valid)))

            # Long pool: pool_multiplier × n_long, capped to half the eligible universe
            # so the long pool can't bleed past the median into short territory.
            # Short side has no stage-2 — bottom n_short by composite directly.
            half_universe = len(valid) // 2
            n_long_pool   = min(half_universe, int(round(n_long * pool_multiplier)))

            # Skip if long pool can't support its final selection or if the universe
            # is too thin to keep long pool and short selection disjoint.
            if n_long_pool < n_long or len(valid) < (n_long_pool + n_short):
                continue

            ranked      = valid.sort_values(ascending=False)
            long_pool   = ranked.index[:n_long_pool].tolist()
            short_coins = ranked.index[-n_short:].tolist()   # bottom n_short by composite

            # ── Stage 2 — refine LONG pool by volume change (high vol_change wins) ──
            # Short leg deliberately skipped — pure composite ranking on the short side.
            if vol_change is not None:
                vc_now     = vol_change.iloc[r]
                long_coins = vc_now[long_pool].sort_values(ascending=False).head(n_long).index.tolist()
                # If vol_change has NaN (early bars), fall back to composite ranking
                if len(long_coins) < n_long: long_coins = long_pool[:n_long]
            else:
                long_coins = long_pool[:n_long]

            universe_size[r] = len(valid)

            if prev_long and prev_short:
                long_turn  = len(set(long_coins)  - set(prev_long))  / n_long
                short_turn = len(set(short_coins) - set(prev_short)) / n_short
            else:
                long_turn  = 1.0
                short_turn = 1.0
            turnover_arr[r]   = long_turn + short_turn
            long_turn_arr[r]  = long_turn
            short_turn_arr[r] = short_turn
            n_long_arr[r]     = len(long_coins)
            n_short_arr[r]    = len(short_coins)

            # ── Vectorised holding-period PnL ─────────────────────────────────
            hold_end  = min(r + K, n)
            long_idx  = [coin_to_idx[c] for c in long_coins]
            short_idx = [coin_to_idx[c] for c in short_coins]

            long_slice  = rets_arr[r:hold_end][:, long_idx]
            short_slice = rets_arr[r:hold_end][:, short_idx]

            # ── Inverse-vol weights at the formation date (bar r-1) ─────────
            # NaN vol → weight 0 (skip).  All-NaN fallback → equal weight.
            # Defensive: never reach inverse-vol with an empty leg
            if len(long_idx) == 0 or len(short_idx) == 0:
                continue

            long_vols  = vol_arr[r - 1][long_idx]
            short_vols = vol_arr[r - 1][short_idx]

            inv_long   = np.where(np.isnan(long_vols)  | (long_vols  == 0), 0.0, 1.0 / long_vols)
            inv_short  = np.where(np.isnan(short_vols) | (short_vols == 0), 0.0, 1.0 / short_vols)
            long_w     = inv_long  / inv_long.sum()  if inv_long.sum()  > 0 else np.full_like(long_vols,  1.0 / len(long_vols))
            short_w    = inv_short / inv_short.sum() if inv_short.sum() > 0 else np.full_like(short_vols, 1.0 / len(short_vols))

            # Weighted basket means with NaN-aware renormalisation per bar:
            # if a coin's return is NaN at bar t, its weight is excluded from
            # the denominator at that bar (rest of the basket fills in pro-rata).
            mask_l       = ~np.isnan(long_slice)
            mask_s       = ~np.isnan(short_slice)
            denom_l      = (mask_l * long_w ).sum(axis=1)
            denom_s      = (mask_s * short_w).sum(axis=1)
            denom_l      = np.where(denom_l == 0, np.nan, denom_l)
            denom_s      = np.where(denom_s == 0, np.nan, denom_s)
            long_basket  = np.nansum(long_slice  * long_w,  axis=1) / denom_l
            short_basket = np.nansum(short_slice * short_w, axis=1) / denom_s

            long_returns[r:hold_end]     = long_basket
            short_returns[r:hold_end]    = short_basket
            strategy_returns[r:hold_end] = 0.5 * long_basket - 0.5 * short_basket

            prev_long  = long_coins
            prev_short = short_coins

        # ── Convert numpy arrays back to pd.Series ───────────────────────────
        # fillna(0) on the return series treats skipped rebalances and warmup
        # bars as 'strategy flat' (no position → 0 return) instead of missing
        # data.  Without this, wf_engine.dropna(subset=['strategy_returns'])
        # deletes the NaN rows, creating K-bar date gaps inside folds wherever
        # the universe filter briefly returned < (n_long + n_short) coins.
        # Warmup bars get filled with 0 too but are removed by the test_start
        # trim in wf_engine, so the net effect on warmup handling is unchanged.
        strategy_returns = pd.Series(strategy_returns, index=dates).fillna(0.0)
        long_returns     = pd.Series(long_returns,     index=dates).fillna(0.0)
        short_returns    = pd.Series(short_returns,    index=dates).fillna(0.0)
        turnover         = pd.Series(turnover_arr,     index=dates)
        universe_size    = pd.Series(universe_size,    index=dates)
        long_turnover    = pd.Series(long_turn_arr,    index=dates)
        short_turnover   = pd.Series(short_turn_arr,   index=dates)
        n_long_held      = pd.Series(n_long_arr,       index=dates)
        n_short_held     = pd.Series(n_short_arr,      index=dates)

        # ── Build output DataFrame ───────────────────────────────────────────
        sr_filled = strategy_returns.fillna(0.0)
        equity    = (1.0 + sr_filled).cumprod()

        result = pd.DataFrame({
            'Open':             equity.shift(1).bfill(),
            'High':             equity,
            'Low':              equity,
            'Close':            equity,
            'Volume':           1.0,
            'position':         1,
            'position_size':    1.0,
            'strategy_returns': strategy_returns,
            'turnover':         turnover,
            'long_ret':         long_returns,
            'short_ret':        short_returns,
            'universe_size':    universe_size,
            'long_turnover':    long_turnover,
            'short_turnover':   short_turnover,
            'n_long_held':      n_long_held,
            'n_short_held':     n_short_held,
        }, index=dates)

        return result, ['strategy_returns']

    return strategy_fn


my_strategy = make_xs_strategy(
    panel            = panel,
    volume           = volume,
    meta             = meta,
    uf_top_n         = UF_TOP_N,
    uf_min_volume    = UF_MIN_VOLUME,
    uf_min_age       = UF_MIN_AGE_DAYS,
    uf_volume_window = UF_VOLUME_WINDOW,
    pool_multiplier  = POOL_MULTIPLIER,
    iv_vol_window    = IV_VOL_WINDOW,
)


---
## Run Walk-Forward

**Fold setup for daily data:**

| Parameter | Value | Rationale |
|---|---|---|
| `TRAIN_BARS` | 1050 | ~3 years; many rebalances per fold at K=7-21 |
| `TEST_BARS` | 285 | ~9.5 months; ~3.7:1 train:test ratio |
| `BURNIN_BARS` | 95 | **MUST be >= 2·max(J) + 1** for the residual-Sharpe signal (double rolling). With max J=45, 91 is the minimum; 95 leaves safety margin. Higher than the single-rolling strategies (xs_momentum / xs_1 / xs_2 use 50). |
| `N_TRIALS` | 300 | 4 free params (J, K, pct_long, pct_short) |
| `cost` | **COST** | Per-leg fraction applied via `turnover × COST` in the engine |

**Why BURNIN_BARS = 95 here:** the residual-Sharpe signal applies TWO rolling(J) operations in sequence — first to compute β to BTC, then to compute the Sharpe of residuals. Each rolling adds J bars of warmup, so `mom_signal` is NaN until bar `2J` (vs `J+1` for the single-rolling strategies). If `BURNIN_BARS < 2J + 1`, the warmup overflows into the test window — and the strategy_fn's `fillna(0)` fix converts those NaN bars into flat 0-return periods at the start of every fold's OOS. Visible as flat equity, 0 turnover, both legs flat. Bumping BURNIN_BARS to 95 fixes it.

**Cost flow:** `strategy_fn` returns gross `strategy_returns` plus a `turnover`
column. The engine computes `trade_cost = turnover × cost` at each rebalance bar.
Costs propagate correctly through Optuna trials, OOS evaluation, plateau/perturbation
analysis, and `cost_stress_test()` without any special handling.

**Custom `reject_fn`:** `num_trades = 0` for this strategy (position=1 everywhere,
so no position changes are detected). The default filter `num_trades < 7` would
reject all trials. Sharpe ≥ 0 and drawdown are used as quality gates instead.

In [ ]:
# ── walk-forward windows ──────────────────────────────────────────────────────
TRAIN_BARS  = 1050   # ~3 years daily
TEST_BARS   = 285    # ~9.5 months
BURNIN_BARS = 95     # xs_3 family uses TWO compounded rolling(J) operations
                     # (rolling β, then rolling Sharpe of residuals), so warmup
                     # is 2*J + 1, not J + 1.  With max J = 45, need >= 91; 95
                     # leaves a small safety margin.  If you lower max J in
                     # PARAM_DEFS, you can drop BURNIN_BARS proportionally.
N_TRIALS    = 300

# ── SCORING FUNCTION ──────────────────────────────────────────────────────────
def score_fn(m):
    SHARPE_MAX = 3.0
    CALMAR_MAX = 6.0
    RETURN_MAX = 10.0

    calmar = (m['total_return'] / abs(m['max_drawdown'])
              if m['max_drawdown'] != 0 else 0.0)

    s = np.clip(m['sharpe_ratio']  / SHARPE_MAX, 0, 1)
    c = np.clip(calmar             / CALMAR_MAX, 0, 1)
    r = np.clip(m['total_return']  / RETURN_MAX, 0, 1)

    return 0.60 * s + 0.25 * c + 0.15 * r


# ── REJECTION CRITERIA ────────────────────────────────────────────────────────
# Do NOT check num_trades — always 0 for this strategy (position=1 everywhere).

def reject_fn(m):
    if m is None:                   return True
    if m['sharpe_ratio'] < 0.0:     return True
    if m['max_drawdown'] < -0.80:   return True
    if m['total_return'] < 0.0:     return True
    return False


results = walk_forward(
    df           = driver_df,
    strategy_fn  = my_strategy,
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    train_bars   = TRAIN_BARS,
    test_bars    = TEST_BARS,
    burnin_bars  = BURNIN_BARS,
    n_trials     = N_TRIALS,
    cost         = COST,       # applied by engine as: trade_cost = turnover * COST
    score_fn     = score_fn,
    reject_fn    = reject_fn,
    save_csv     = None,
)


---
## Granular Results and Parameter Stability

Per-fold IS vs OOS performance. Compare `train_*` vs `test_*` to assess overfitting.

| Column | Description |
|---|---|
| `*_sharpe` `*_return` `*_drawdown` `*_calmar` | Core performance metrics |
| `optuna_score` | Best score achieved on training window |
| `param_J` `param_K` `param_pct_long` `param_pct_short` | Best parameter values per fold |

**Note:** `test_trades`, `test_winrate`, `test_profit_factor` will all be 0 — the
engine logs no trades because `position = 1` everywhere (see Cell 4). Use Sharpe,
return, drawdown, **and the long/short attribution table further below** as the
primary OOS quality signals for this strategy.

**Consensus Parameters** — anchor parameters with CV < 0.15 in `FIXED_PARAMS`.
For percentile-based XS momentum, `pct_long` and `pct_short` are often the most
stable since the strategy is now scale-invariant to universe size.

In [ ]:
# ── fold summary table ────────────────────────────────────────────────────────
display_cols = [
    'train_start', 'train_end',
    'test_start',  'test_end',
    'train_sharpe', 'test_sharpe',
    'train_return', 'test_return',
    'train_drawdown', 'test_drawdown',
    'optuna_score',
    'param_J', 'param_K', 'param_pct_long', 'param_pct_short',
]
display(results['results_df'][display_cols].round(3))

# ── parameter stability ───────────────────────────────────────────────────────
stability = results['stability_df'].copy()
stability['stable'] = stability['stable'].map({True: 'yes', False: ''})
stability['fixed']  = stability['fixed'].map({True: 'yes', False: ''})
stability = stability[['param', 'median', 'std', 'cv', 'stable', 'fixed']].round(3)
print('\nParameter stability (CV < 0.15 = stable candidate for fixing):')
display(stability.sort_values('cv'))

# ── consensus params ──────────────────────────────────────────────────────────
stable = results['stability_df'][results['stability_df']['cv'] < 0.15]

print('Stable parameters (CV < 0.15) — copy into FIXED_PARAMS:')
for _, row in stable.iterrows():
    v = results['consensus_params'][row['param']]
    v_fmt = (int(round(v)) if isinstance(v, float) and v == int(v)
             else round(v, 4) if isinstance(v, float) else v)
    print(f"    '{row['param']}': {v_fmt},")

print('\nConsensus parameters (median across folds):')
for k, v in results['consensus_params'].items():
    print(f'  {k:<12} = {round(v, 4) if isinstance(v, float) else v}')

---
## Parameter Robustness Checks

### Plateau Analysis
Sweeps each free parameter across its full range (holding others at consensus),
evaluating the score at each point over the full date range.

| Verdict | Plateau % | Meaning |
|---|---|---|
| Robust | ≥ 60% | Score stays near peak across most of the range |
| Moderate | 30–60% | Some sensitivity — watch if it degrades further after fixing |
| FRAGILE | < 30% | Strategy depends critically on hitting the exact value — consider anchoring |
| N/A | — | Every sweep point failed reject_fn — strategy completely intolerant on this axis |

### Perturbation Test
Randomly jitters **all** free parameters simultaneously by ±5/10/20% of their range.
Tests whether the optimum is a broad hill or a narrow spike in the joint parameter space.

> With only 3 integer parameters, this test is less informative than for 15-param
> strategies. Focus on the plateau % instead.

In [ ]:
# ── 1-D sensitivity sweeps around consensus params ─────────────────────────
sweep_results = plateau_analysis(
    df           = driver_df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = 0.0,         # costs baked into strategy_returns
    reject_fn    = reject_fn,   # required: default reject_fn checks num_trades < 7 which is always 0
    n_steps      = 20,
)

# ── text verdicts ──────────────────────────────────────────────────────────
verdict_df = plateau_summary(
    sweep_results,
    base_params  = results['consensus_params'],
    stability_df = results['stability_df'],
    threshold    = 0.20,
)

# ── neighbourhood perturbation ─────────────────────────────────────────────
# With only 3 integer parameters the perturbation may hit boundary values often.
# Treat degradation > 20% at ±10% offset as a fragility signal.
perturb_df = perturbation_test(
    df           = driver_df,
    strategy_fn  = my_strategy,
    base_params  = results['consensus_params'],
    param_defs   = PARAM_DEFS,
    fixed_params = FIXED_PARAMS,
    cost         = 0.0,
    reject_fn    = reject_fn,   # required: default reject_fn checks num_trades < 7 which is always 0
    pct_offsets  = (0.05, 0.10, 0.20),
    n_samples    = 50,
)

In [ ]:

# ── plateau sweep charts ──────────────────────────────────────────────────────
plot_plateau_analysis(
    sweep_results    = sweep_results,
    consensus_params = results['consensus_params'],
    param_defs       = PARAM_DEFS,
    fixed_params     = FIXED_PARAMS,
    threshold        = 0.20,
    show             = False,
    save_html        = None,
)


---
## Results Charts and Cost Stress Test

**OOS equity chart** — combined OOS period stitched from all folds, overlaid with the
equal-weight universe basket (benchmark).

**Why the equal-weight basket is the right benchmark for this strategy:**  
A dollar-neutral XS momentum portfolio (long top-N, short bottom-N, equal weight) has
**zero net market exposure** by construction — the long and short legs cancel in aggregate.
This means BTC buy-and-hold is *not* a valid comparison: if the whole market rises uniformly,
a dollar-neutral strategy should earn approximately zero from that move.

The equal-weight basket represents the passive return from holding all universe coins equally,
with no selection or ranking. Any outperformance over this baseline is *pure cross-sectional
alpha* — return that comes solely from the ranking signal correctly identifying relative
winners and losers, not from riding overall market direction.

Reading the chart:
- **Benchmark rises, strategy lags** → the short leg is the drag; you are short coins that
  are rising, just slower than the longs. The strategy is still doing its job, but market
  beta inside the long leg does not offset the short-leg drag.
- **Benchmark falls, strategy holds or rises** → the short leg is working; the coins you
  are short are falling harder than the longs, generating positive spread return.
- **Both fall together** → a broad market crash overwhelms cross-sectional differentiation;
  expected for XS momentum in high-correlation bear regimes where all assets move together.

**Cost stress test** — re-runs the combined OOS backtest at escalating cost multipliers.
Works correctly because `oos_combined_df` carries the `turnover` column: the engine applies
`trade_cost = turnover × cost` at each rebalance bar, so raising `cost` correctly scales
the burden on high-turnover periods without re-running the strategy.

In [ ]:
# ── OOS equity + fold charts ──────────────────────────────────────────────────
plot_walk_forward_results(
    results         = results,
    param_defs      = PARAM_DEFS,
    fixed_params    = FIXED_PARAMS,
    benchmark_data  = ew_df,       # equal-weight universe basket
    show            = True,
    save_html_dir   = None,
    show_fold_perf  = False,
    show_param_evol = False,
    show_oos_equity = True,
    show_trades     = False,
)

# ── transaction cost stress test ──────────────────────────────────────────────
if results['oos_combined_df'] is not None:
    cost_df = cost_stress_test(
        oos_combined_df  = results['oos_combined_df'],
        cost_multipliers = (1.0, 1.5, 2.0, 3.0),
        base_cost        = COST,
    )
else:
    print('No combined OOS dataframe — skip cost stress test.')

---
## Long/Short Attribution Diagnostics

Cross-sectional-specific diagnostics that replace trade-centric metrics
(meaningless here since `position=1` always — the engine logs zero trades).

**Attribution table** — Sharpe and total return decomposed into:
- `long_sharpe` / `long_total_return` — performance of being 100% long the long basket
- `short_sharpe` / `short_basket_total` — performance of being 100% short the short basket
- `spread_sharpe` / `spread_total` — dollar-neutral spread (long − short, 1× leverage)
- `net_sharpe` — combined post-cost return (matches the OOS Sharpe in the fold table)
- `hit_rate` — fraction of rebalance periods where (long − short) > 0
- `mean_turnover` — average fractional turnover at rebalance bars (range 0–2)
- `avg/min/max_universe_size` — # coins available at rebalance over the OOS period

**Single most important read:** if `short_sharpe < 0` consistently across runs, the
short leg is a drag rather than a profit centre. That argues for dropping it
(long-only) or re-thinking the ranking (the bottom of the universe might not be
mean-reverting losers — it might be coins that already crashed and bounce).

**4-panel chart** —
1. Cumulative equity by leg (long, −short, combined net) — visual long/short attribution
2. Cumulative spread (long − short) — the raw momentum premium harvested
3. Turnover per rebalance — high turnover = unstable rankings = noisy signal
4. Universe size over time — flags degraded regimes where ranking is meaningless

In [ ]:
# ── long/short attribution table + diagnostic chart ──────────────────────────
oos = results['oos_combined_df']

if oos is None or 'long_ret' not in oos.columns:
    print('No OOS dataframe with attribution columns — skip diagnostics.')
else:
    attr = compute_attribution(oos, ann=365)

    print('═' * 60)
    print('LONG/SHORT ATTRIBUTION')
    print('═' * 60)
    for k, v in attr.items():
        if isinstance(v, float):
            if 'sharpe' in k:
                print(f'  {k:<22} {v:>8.3f}')
            elif 'return' in k or 'total' in k or 'rate' in k or 'turnover' in k:
                print(f'  {k:<22} {v:>8.2%}' if abs(v) < 100 else f'  {k:<22} {v:>8.3f}')
            else:
                print(f'  {k:<22} {v:>8.2f}')
        else:
            print(f'  {k:<22} {v:>8}')

    plot_attribution(oos,
                     benchmark_data=ew_df,   # equal-weight basket → strips beta from leg returns
                     show=True, save_html=None,
                     title='XS Momentum — Long/Short Diagnostics (OOS)')

---
## Bear-Hedge Diagnostic — Conditional Regime Performance

Tests whether this XS strategy genuinely works as a hedge to long-only /
momentum strategies during BTC drawdowns. Splits OOS bars by sign of BTC's
daily return and computes per-regime alphas + strategy Sharpe.

**What to read:**
- `short_α(down)` > `short_α(up)` → short leg generates real return when BTC
  is falling. **This is the bear-hedge condition.**
- `strat_Sharpe(down)` > 0 → strategy makes money in BTC-down regimes. If
  this is positive while your other momentum strategies struggle in those
  same windows, XS earns its slot in the portfolio.
- `|corr(strategy, BTC)|` near 0 → dollar-neutral construction is holding,
  no hidden BTC beta leaking through.

**If short_α(down) ≤ short_α(up):** the short leg is *not* a hedge — it's just
correlated alpha. Either redesign the short signal or accept the strategy as
a separate alpha source rather than a hedge. **Don't asymmetrically size
toward the short leg until this test confirms it works in BTC-down.**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Bear-hedge diagnostic — performance conditional on BTC regime
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np

oos = results['oos_combined_df']
ANN = 365   # daily crypto annualisation

# BTC return series aligned to OOS index. Pulled from the panel used for PnL,
# so this is internally consistent with the strategy's own return computation.
btc_ret = panel['BTC'].pct_change().reindex(oos.index).fillna(0)

up_mask   = btc_ret > 0
flat_mask = btc_ret == 0
down_mask = btc_ret < 0


def _stats(label, mask):
    n = int(mask.sum())
    if n == 0:
        print(f'  {label:<18}  bars=   0  (no data)')
        return
    long_alpha   = (oos['long_ret']  - btc_ret)[mask].mean() * ANN
    short_alpha  = (btc_ret - oos['short_ret'])[mask].mean() * ANN
    strat_mean   = oos['strategy_returns'][mask].mean() * ANN
    strat_std    = oos['strategy_returns'][mask].std() * np.sqrt(ANN)
    strat_sharpe = strat_mean / strat_std if strat_std > 0 else 0.0
    print(f'  {label:<18}  bars={n:>4}  '
          f'long_α={long_alpha*100:>+7.1f}%  short_α={short_alpha*100:>+7.1f}%  '
          f'strat_Sharpe={strat_sharpe:>+5.2f}')


print('═' * 86)
print('BEAR-HEDGE DIAGNOSTIC — Performance by BTC regime  (annualised)')
print('═' * 86)
print(f'{"Regime":<20}{"Bars":>6}  {"Long alpha vs BTC":>18}  '
      f'{"Short alpha vs BTC":>20}  {"Strat Sharpe":>14}')
print('─' * 86)
_stats('BTC up days',   up_mask)
_stats('BTC down days', down_mask)
_stats('All bars',      pd.Series(True, index=oos.index))

# Full-period correlation — confirms dollar-neutral construction
corr = oos['strategy_returns'].corr(btc_ret)
if abs(corr) < 0.15:
    corr_label = 'dollar-neutral holding'
elif corr > 0:
    corr_label = 'POSITIVE BTC beta leakage'
else:
    corr_label = 'NEGATIVE BTC tilt (net short bias)'
print(f'\ncorr(strategy_returns, BTC) = {corr:+.3f}   ({corr_label})')

# Hedge condition summary — print a single verdict line
sa_up   = ((btc_ret - oos['short_ret']) * up_mask).sum()   / max(up_mask.sum(),   1) * ANN
sa_down = ((btc_ret - oos['short_ret']) * down_mask).sum() / max(down_mask.sum(), 1) * ANN
hedge_works = (sa_down > sa_up) and (sa_down > 0)
print(f'\nVerdict: {"✓ short leg behaves like a bear hedge" if hedge_works else "✗ short leg does NOT add value in BTC-down regimes"}')
print(f'         (short_α: BTC-up = {sa_up*100:+.1f}%, BTC-down = {sa_down*100:+.1f}%)')

---
## Regime Quadrant Diagnostic — BTC Trend × Trend Strength

Splits the OOS period into four regime quadrants and reports strategy performance
in each. Reveals where the strategy makes its money vs where it bleeds — a
prerequisite to designing regime-conditional filters.

**Two regime axes:**

| | Bull *(BTC > 200d MA)* | Bear *(BTC < 200d MA)* |
|---|---|---|
| **Trending** *(ER ≥ median)* | Bull / trending | Bear / trending |
| **Chop**  *(ER < median)*    | Bull / chop      | Bear / chop      |

**Trend-strength = Kaufman Efficiency Ratio (14-day)** on BTC close:
`ER = |close[t] − close[t−14]| / Σ|close[i] − close[i−1]|`. Range [0, 1]; high
= clean directional move, low = same net move achieved through choppy back-and-
forth. Conceptually equivalent to ADX (which also measures trend persistence)
but computable from Close alone — the cache doesn't store H/L, so true ADX
would require a cache rebuild. ER captures ~95% of the same regime signal.

**What to read:**
- **Returns positive in 3 quadrants, negative in 1** → high-value regime filter
  candidate (turn strategy off in the bad quadrant).
- **Returns negative in ≥2 quadrants** → strategy is fragile across regimes;
  filter alone won't fix it, signal needs work.
- **Bear-hedge use case specifically:** want positive Sharpe in *both* bear
  quadrants. Bear/trending only → strategy is a bear-trend hedge (works in
  waterfall declines, fails in slow grinders). Bear/chop only → strategy is a
  bear-mean-reversion hedge (works in choppy bears, fails when BTC capitulates).
- **Bull/trending dominant return** → the strategy is essentially a long-only
  momentum bet riding bull trends. Not a hedge; a directional alpha source.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Regime quadrant diagnostic — BTC trend × trend-strength (Efficiency Ratio)
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

oos = results['oos_combined_df']
ANN = 365

# ── BTC trend regime: above/below 200-day MA ─────────────────────────────────
btc        = panel['BTC']
btc_ma200  = btc.rolling(200).mean()
bull_mask  = (btc > btc_ma200).reindex(oos.index).fillna(False)

# ── Trend-strength regime: Kaufman Efficiency Ratio (14-day) ────────────────
# Close-only analog of ADX.  Range [0, 1].  High = directional, low = chop.
ER_WIN     = 14
direction  = btc.diff(ER_WIN).abs()
path_total = btc.diff().abs().rolling(ER_WIN).sum()
er_full    = (direction / path_total.replace(0, np.nan)).fillna(0)
er         = er_full.reindex(oos.index)

# Threshold by median over OOS window — adaptive to data, no magic number
er_pos     = er[er > 0]
er_thresh  = float(er_pos.median()) if len(er_pos) > 0 else 0.3
trend_mask = er >= er_thresh

# ── Build quadrant labels ────────────────────────────────────────────────────
quadrant = pd.Series('—', index=oos.index, dtype='object')
quadrant.loc[ bull_mask &  trend_mask] = 'Bull / trending'
quadrant.loc[ bull_mask & ~trend_mask] = 'Bull / chop'
quadrant.loc[~bull_mask &  trend_mask] = 'Bear / trending'
quadrant.loc[~bull_mask & ~trend_mask] = 'Bear / chop'

# ── Per-quadrant stats ───────────────────────────────────────────────────────
print('═' * 86)
print('REGIME QUADRANT DIAGNOSTIC — performance by BTC trend × trend-strength')
print('═' * 86)
print(f'BTC trend split:      bull = above 200d MA   bear = below 200d MA')
print(f'Trend-strength split: median Efficiency Ratio over OOS = {er_thresh:.3f}')
print()
print(f'{"Quadrant":<22}{"% OOS":>9}{"Bars":>7}{"Ann return":>14}'
      f'{"Sharpe":>10}{"Total return":>16}')
print('─' * 86)
for q in ['Bull / trending', 'Bull / chop', 'Bear / trending', 'Bear / chop']:
    mask = quadrant == q
    n    = int(mask.sum())
    if n == 0:
        print(f'  {q:<20}{"—":>9}{0:>7}{"—":>14}{"—":>10}{"—":>16}')
        continue
    pct      = n / len(oos) * 100
    rets     = oos['strategy_returns'][mask]
    ann_ret  = rets.mean() * ANN
    ann_vol  = rets.std()  * np.sqrt(ANN)
    sharpe   = ann_ret / ann_vol if ann_vol > 0 else 0.0
    tot_ret  = (1 + rets).prod() - 1
    print(f'  {q:<20}{pct:>8.1f}%{n:>7}{ann_ret*100:>+13.1f}%{sharpe:>+10.2f}'
          f'{tot_ret*100:>+15.1f}%')

# ── Verdict line ────────────────────────────────────────────────────────────
quadrant_returns = {
    q: float((1 + oos['strategy_returns'][quadrant == q]).prod() - 1)
    for q in ['Bull / trending', 'Bull / chop', 'Bear / trending', 'Bear / chop']
}
n_neg = sum(1 for v in quadrant_returns.values() if v < 0)
print()
if n_neg == 0:
    print('Verdict: ✓ strategy positive in all four quadrants — no regime filter needed')
elif n_neg == 1:
    bad = next(q for q, v in quadrant_returns.items() if v < 0)
    print(f'Verdict: regime filter candidate — turn strategy OFF in {bad!r}')
elif n_neg == 2:
    bad = [q for q, v in quadrant_returns.items() if v < 0]
    print(f'Verdict: strategy struggles in {bad}; consider signal redesign over filter')
else:
    print('Verdict: strategy is fragile across regimes — filter alone unlikely to help')

# ── Chart: equity + regime quadrant timeline ────────────────────────────────
equity = (1 + oos['strategy_returns'].fillna(0)).cumprod().clip(lower=0)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.62, 0.38], vertical_spacing=0.08,
                    subplot_titles=('Strategy equity (clipped at 0)',
                                    'BTC regime quadrant per OOS bar'))

fig.add_trace(go.Scatter(x=equity.index, y=equity.values, name='Strategy',
                          line=dict(color='black', width=2), showlegend=False),
              row=1, col=1)
fig.add_hline(y=1.0, line=dict(color='gray', dash='dash'), row=1, col=1)

quadrant_levels = {
    'Bull / trending': 4, 'Bull / chop': 3,
    'Bear / trending': 2, 'Bear / chop': 1,
}
quadrant_colors = {
    'Bull / trending': '#1abc9c',   # teal-green (clean bull)
    'Bull / chop':     '#a3e4b8',   # pale green
    'Bear / trending': '#c0392b',   # deep red (clean bear)
    'Bear / chop':     '#f5a8a0',   # pale red
}
for q, lvl in quadrant_levels.items():
    mask = quadrant == q
    if mask.any():
        fig.add_trace(go.Scatter(
            x=oos.index[mask], y=[lvl] * int(mask.sum()),
            mode='markers', marker=dict(color=quadrant_colors[q], size=5),
            name=q, hovertemplate=f'{q}<br>%{{x|%Y-%m-%d}}<extra></extra>',
        ), row=2, col=1)

fig.update_yaxes(title_text='Equity', row=1, col=1)
fig.update_yaxes(title_text='Regime', row=2, col=1,
                 tickvals=[1, 2, 3, 4],
                 ticktext=['Bear / chop', 'Bear / trending',
                          'Bull / chop',  'Bull / trending'])
fig.update_layout(height=620, hovermode='x unified',
                  title='Strategy equity vs BTC regime quadrants',
                  legend=dict(orientation='h', yanchor='bottom',
                              y=1.02, xanchor='left', x=0))
fig.show()

---
## Live Execution Notes (forward-looking — for when this goes live)

**Venue:** Binance (single account, no multi-venue complexity).

**Recommended structure:** spot-long + perp-short under cross-margin.
- **Long leg** → Binance spot (just hold the coin)
- **Short leg** → Binance USDT-M perp (open short position)
- Cross-margin lets the spot holdings collateralize the perp shorts → capital efficiency stays close to perp/perp.

**Why this beats perp/perp:**
On XS momentum specifically, the long leg picks the *highest-momentum* coins, which systematically have the *highest* funding rates (long-biased flow). The short leg picks the *lowest-momentum* coins, which usually have low / negative funding rates. So perp/perp pays a lot on longs and only partially offsets via shorts. Spot-long avoids the long-side cost entirely; perp-short still receives any short-side funding. Net funding moves from "drag" to "small positive."

**Funding-rate cache deferred.** With this structure funding is no longer a first-order cost — modeling it precisely is a refinement, not a gate. Build it later only if you also want to backtest a perp/perp variant for comparison. Architecture sketch (when needed):
- `infrastructure/data/funding_cache.py` — mirror `universe_cache.py` (incremental fetch, parquet output)
- `infrastructure/perps/funding.py` — `apply_funding(positions, funding_panel)` helper
- Sign convention: `funding_pnl = -position × funding_rate` (long pays when rate > 0)

**Backtest data note:** the perp cache built by `universe_cache.py` is fine as a price proxy for spot in the backtest. For top-30 liquid Binance coins, spot ↔ perp basis runs ~5–15 bps intraday, well under the noise floor of a daily strategy. Don't build a parallel spot cache just for backtesting purposes.

**Bigger missing items to model before going live (in priority order):**
1. **Slippage on the bottom of the universe.** Rank 25–30 by volume can be thinly traded — assume 5–15 bps per leg vs the close.
2. **Borrow availability** if you ever go spot-margin-short instead of perp-short (some coins are hard-to-borrow; rates spike in stressed regimes).
3. **Funding** — only after the above two are calibrated. For spot-long + perp-short it's a small positive contribution, not a drag.

In [ ]:
# ── save OOS dataframe ────────────────────────────────────────────────────────
# Load in a portfolio notebook with:
#   xs_oos = pd.read_pickle(os.path.join('oos', 'xs_3_3_i_oos.pkl'))
if results['oos_combined_df'] is not None:
    oos_dir  = os.path.join(os.path.dirname(os.path.abspath('xs_momentum_wf.ipynb')), 'oos')
    os.makedirs(oos_dir, exist_ok=True)
    oos_path = os.path.join(oos_dir, 'xs_3_3_i_oos.pkl')
    results['oos_combined_df'].to_pickle(oos_path)
    print(f'Saved OOS dataframe  ({len(results["oos_combined_df"])} bars)  →  {oos_path}')
else:
    print('No combined OOS dataframe to save.')
